# Bronze — Yahoo Finance Ingestion

Fetches daily market data from 2022-01-01 to today for ISK exchange rates and the OMX Iceland All-Share Index.

**Source:** Yahoo Finance via `yfinance`  
**Tickers:** `EURUSD=X`, `ISKUSD=X`, `^OMX`  
**Output:** `bronze.yahoo_finance_raw` (Delta table)  
**Schedule:** Daily  

In [ ]:
import yfinance as yf
import pandas as pd
import re

In [ ]:
TICKERS = ["EURUSD=X", "ISKUSD=X", "^OMX"]
INTERVAL = "1d"

raw_data = yf.download(
    tickers=TICKERS,
    start="2022-01-01",
    interval=INTERVAL,
    auto_adjust=True
)

# Flatten MultiIndex columns (price_type, ticker) and sanitise names
raw_data.columns = [
    re.sub(r'[^a-zA-Z0-9]', '_', f"{price}_{ticker}").strip('_').lower()
    for price, ticker in raw_data.columns
]

raw_data = raw_data.reset_index()
raw_data.columns = [
    re.sub(r'[^a-zA-Z0-9]', '_', c).strip('_').lower()
    for c in raw_data.columns
]

raw_data['ingested_at'] = pd.Timestamp.now()

print(f"Fetched {len(raw_data)} rows | Columns: {raw_data.columns.tolist()}")

if len(raw_data) == 0:
    raise ValueError("[bronze_yahoo_finance] yfinance returned 0 rows - halting.")

In [ ]:
spark_df = spark.createDataFrame(raw_data)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.yahoo_finance_raw")

print(f"Saved to bronze.yahoo_finance_raw — {spark_df.count()} rows")